In [ ]:
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("words", quiet=True)

True

In [1]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

original_poem = """One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is."""

# 1. Load Model and Tokenizer
model_name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()


def get_replacement_word(tokenizer, model, sentence_prefix, word_index):
    """Generates the next token prediction for the given prefix and returns the

    token at the specified index from sorted logits.
    """
    try:
        input_ids = tokenizer.encode(sentence_prefix, return_tensors="pt")

        with torch.no_grad():
            outputs = model(input_ids)
            predictions = outputs.logits

        next_token_logits = predictions[0, -1, :]
        sorted_logits, sorted_indices = torch.sort(
            next_token_logits, descending=True
        )

        if word_index < len(sorted_indices):
            next_token_id = sorted_indices[word_index].item()
            return tokenizer.decode(next_token_id).strip()
        else:
            next_token_id = sorted_indices[0].item()
            return tokenizer.decode(next_token_id).strip()
    except Exception as e:
        print(f"Error generating content: {e}")
        return "[error]"


def generate_poem_file(poem_text, word_index, output_filename):
    modified_lines = []

    for line in poem_text.split("\n"):
        words = line.split()
        if not words:
            modified_lines.append(line)
            continue

        original_last_word_full = words[-1]

        # Extract trailing punctuation
        match = re.match(r"(\w*)([.,;!?'\"]*)$", original_last_word_full)
        trailing_punctuation = match.group(2) if match else ""

        sentence_prefix_for_model = " ".join(words[:-1])

        # Get replacement token using raw index
        new_last_word = get_replacement_word(
            tokenizer, model, sentence_prefix_for_model, word_index
        )

        new_last_word_with_punct = new_last_word + trailing_punctuation

        if sentence_prefix_for_model:
            modified_line = f"{sentence_prefix_for_model} {new_last_word_with_punct}"
        else:
            modified_line = new_last_word_with_punct

        modified_lines.append(modified_line)

    result_poem = "\n".join(modified_lines)

    # Save deliverable file
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(result_poem)

    return result_poem


# --- 1. Generate P+7 (7th token -> index 6) ---
p7_output = generate_poem_file(
    original_poem, word_index=6, output_filename="P+7.txt"
)
print("=== Modified Poem (P+7) ===")
print(p7_output)

# --- 2. Generate P+x (e.g. 24th token -> index 23) ---
x_index = 23
px_output = generate_poem_file(
    original_poem, word_index=x_index, output_filename=f"P+{x_index+1}.txt"
)
print(f"\n=== Modified Poem (P+{x_index+1}) ===")
print(px_output)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

=== Modified Poem (P+7) ===
One must have a mind of her
To regard the frost and the death
Of the pine-trees crusted with oil;
And have been cold a long way
To behold the junipers shagged with white,
The spruces rough in the distant horizon
Of the January sun; and not to have
Of any misery in the sound of the sound,
In the sound of a few shots,
Which is the sound of the voice
Full of the same day
That is blowing in the same bare air
For the listener, who listens in the morning,
And, nothing himself, I
Nothing that is not there and the nothing that isn.

=== Modified Poem (P+24) ===
One must have a mind of thy
To regard the frost and the lightning
Of the pine-trees crusted with sugar;
And have been cold a long times
To behold the junipers shagged with venom,
The spruces rough in the distant forests
Of the January sun; and not to this
Of any misery in the sound of the news,
In the sound of a few cheers,
Which is the sound of the explosion
Full of the same in
That is blowing in the same ba